# 25: Transformer Architecture - Putting It All Together

## The Complete Picture

Now we combine everything we've learned:
- ✅ Self-attention
- ✅ Multi-head attention
- ✅ Positional encoding
- ✅ Feed-forward networks

Into the **Transformer**: the architecture that revolutionized NLP!

### The Web Dev Analogy

The Transformer is like **microservices architecture**:
- **Encoder**: Process input (like REST API ingestion)
- **Decoder**: Generate output (like response generation)
- **Attention**: Services communicate directly (no bottleneck)
- **Parallel**: All services run simultaneously (not sequential)
- **Scalable**: Easy to add more layers/heads

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

print("Ready to build Transformers! 🤖")

## 1. Transformer Overview

**Architecture:**
```
Input → Encoder → Decoder → Output
```

**Key Components:**
1. **Encoder**: Processes input sequence
   - Multi-head self-attention
   - Feed-forward network
   - Layer normalization
   - Residual connections

2. **Decoder**: Generates output sequence
   - Masked self-attention (can't see future)
   - Cross-attention to encoder output
   - Feed-forward network
   - Layer normalization
   - Residual connections

In [ ]:
# Visualize architecture
print("Transformer Architecture:")
print("=" * 70)
print("\nENcoDER STACK (N layers):")
print("  Input Embedding + Positional Encoding")
print("  ↓")
print("  [Multi-Head Self-Attention → Add & Norm]")
print("  ↓")
print("  [Feed-Forward → Add & Norm]")
print("  ↓")
print("  ... (repeat N times)")
print("  ↓")
print("  Encoder Output")

print("\nDECODER STACK (N layers):")
print("  Output Embedding + Positional Encoding")
print("  ↓")
print("  [Masked Multi-Head Self-Attention → Add & Norm]")
print("  ↓")
print("  [Multi-Head Cross-Attention (to Encoder) → Add & Norm]")
print("  ↓")
print("  [Feed-Forward → Add & Norm]")
print("  ↓")
print("  ... (repeat N times)")
print("  ↓")
print("  Linear + Softmax")
print("  ↓")
print("  Output Probabilities")

`★ Insight ─────────────────────────────────────`

**Key Transformer innovations:**
1. **No recurrence**: Parallel processing (faster training)
2. **Self-attention**: Direct connections between all positions
3. **Residual connections**: Enable very deep networks
4. **Layer norm**: Stable training
5. **Multi-head**: Multiple attention perspectives

These enable models with billions of parameters!

`─────────────────────────────────────────────────`

## 2. Building Blocks

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head attention mechanism."""
    
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        # Linear projections in batch
        Q = self.W_q(query)  # (batch, seq_len, d_model)
        K = self.W_k(key)
        V = self.W_v(value)
        
        # Split into multiple heads
        # (batch, seq_len, d_model) → (batch, seq_len, num_heads, d_k)
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        # Now: (batch, num_heads, seq_len, d_k)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # Apply attention to values
        output = torch.matmul(attention_weights, V)
        # (batch, num_heads, seq_len, d_k)
        
        # Concatenate heads
        output = output.transpose(1, 2).contiguous()
        output = output.view(batch_size, -1, self.d_model)
        
        # Final linear projection
        output = self.W_o(output)
        
        return output, attention_weights

# Test
mha = MultiHeadAttention(d_model=64, num_heads=8)
x = torch.randn(2, 10, 64)  # (batch, seq_len, d_model)
output, attn = mha(x, x, x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention shape: {attn.shape}")
print(f"  (batch, num_heads, seq_len, seq_len)")

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network."""
    
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = self.linear1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

# Test
ff = FeedForward(d_model=64, d_ff=256)
x = torch.randn(2, 10, 64)
output = ff(x)

print(f"Feed-Forward:")
print(f"  Input shape: {x.shape}")
print(f"  Output shape: {output.shape}")
print(f"\n💡 Applied independently to each position!")

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""
    
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * 
            (-math.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

## 3. Encoder Layer

In [ ]:
class EncoderLayer(nn.Module):
    """Single Transformer encoder layer."""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # Multi-head attention
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Feed-forward
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Dropout
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention + residual + norm
        attn_output, _ = self.self_attn(x, x, x, mask)
        x = x + self.dropout1(attn_output)  # Residual
        x = self.norm1(x)  # Layer norm
        
        # Feed-forward + residual + norm
        ff_output = self.feed_forward(x)
        x = x + self.dropout2(ff_output)  # Residual
        x = self.norm2(x)  # Layer norm
        
        return x

# Test
encoder_layer = EncoderLayer(d_model=64, num_heads=8, d_ff=256)
x = torch.randn(2, 10, 64)
output = encoder_layer(x)

print(f"Encoder Layer:")
print(f"  Input shape: {x.shape}")
print(f"  Output shape: {output.shape}")
print(f"\n✅ Shape preserved through layer!")

`★ Insight ─────────────────────────────────────`

**Why residual connections + layer norm:**
1. **Residual**: `x + F(x)` allows gradients to flow directly
2. **Layer norm**: Stabilizes activations, faster training
3. **Together**: Enable very deep networks (100+ layers possible)

Without these, deep Transformers would be unstable!

`─────────────────────────────────────────────────`

## 4. Full Encoder

In [ ]:
class TransformerEncoder(nn.Module):
    """Stack of N encoder layers."""
    
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        
        # Embedding
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)
        
        # Stack of encoder layers
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.d_model = d_model
    
    def forward(self, x, mask=None):
        # x: (batch, seq_len) - token indices
        
        # Embedding + scaling
        x = self.embedding(x) * math.sqrt(self.d_model)
        
        # Positional encoding
        x = self.pos_encoding(x)
        
        # Pass through encoder layers
        for layer in self.layers:
            x = layer(x, mask)
        
        return self.norm(x)

# Test
encoder = TransformerEncoder(
    vocab_size=1000,
    d_model=64,
    num_heads=8,
    d_ff=256,
    num_layers=6,
    dropout=0.1
)

# Input: token indices
input_tokens = torch.randint(0, 1000, (2, 10))  # (batch, seq_len)
output = encoder(input_tokens)

print(f"Full Encoder:")
print(f"  Input tokens shape: {input_tokens.shape}")
print(f"  Output shape: {output.shape}")
print(f"  (batch, seq_len, d_model)")
print(f"\nTotal parameters: {sum(p.numel() for p in encoder.parameters()):,}")

## 5. Decoder Layer (with Masking)

In [ ]:
class DecoderLayer(nn.Module):
    """Single Transformer decoder layer."""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # Masked self-attention
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Cross-attention to encoder output
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Feed-forward
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        
        # Layer norm
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        
        # Dropout
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # Masked self-attention
        attn_output, _ = self.self_attn(x, x, x, tgt_mask)
        x = x + self.dropout1(attn_output)
        x = self.norm1(x)
        
        # Cross-attention to encoder
        attn_output, _ = self.cross_attn(x, encoder_output, encoder_output, src_mask)
        x = x + self.dropout2(attn_output)
        x = self.norm2(x)
        
        # Feed-forward
        ff_output = self.feed_forward(x)
        x = x + self.dropout3(ff_output)
        x = self.norm3(x)
        
        return x

print("Decoder Layer has 3 sub-layers:")
print("  1. Masked self-attention (can't see future)")
print("  2. Cross-attention (attends to encoder output)")
print("  3. Feed-forward")
print("\nEach with residual connection + layer norm!")

## 6. Creating Masks

In [ ]:
def create_causal_mask(size):
    """Create mask to prevent attention to future positions."""
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    return mask == 0  # True where we CAN attend

# Visualize
mask = create_causal_mask(5)

plt.figure(figsize=(8, 6))
plt.imshow(mask.numpy(), cmap='RdYlGn', aspect='auto')
plt.colorbar(label='Can Attend', ticks=[0, 1])
plt.xlabel('Key Position', fontsize=12)
plt.ylabel('Query Position', fontsize=12)
plt.title('Causal (Look-Ahead) Mask', fontsize=14, fontweight='bold')
plt.xticks(range(5))
plt.yticks(range(5))

# Add grid
for i in range(6):
    plt.axhline(i-0.5, color='black', linewidth=0.5)
    plt.axvline(i-0.5, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

print("\n💡 Causal mask ensures decoder can only attend to previous positions!")
print("   Position 0: can only see position 0")
print("   Position 1: can see positions 0-1")
print("   Position 2: can see positions 0-2")
print("   etc.")

## 7. Model Variants

In [ ]:
print("Transformer Variants:")
print("=" * 70)

print("\n1. Encoder-Decoder (Original Transformer):")
print("   - Full encoder + decoder")
print("   - Use case: Translation, summarization")
print("   - Example: T5, BART")

print("\n2. Encoder-Only:")
print("   - Only encoder layers")
print("   - Bidirectional attention (sees all tokens)")
print("   - Use case: Classification, understanding")
print("   - Example: BERT, RoBERTa")

print("\n3. Decoder-Only:")
print("   - Only decoder layers (with causal masking)")
print("   - Autoregressive (left-to-right)")
print("   - Use case: Text generation, completion")
print("   - Example: GPT-2, GPT-3, GPT-4")

print("\n" + "=" * 70)
print("Modern trend: Decoder-only models dominate!")
print("  - Simpler architecture")
print("  - Scales to huge sizes")
print("  - Versatile (can do many tasks)")

## 8. Key Hyperparameters

In [ ]:
print("Transformer Hyperparameters:")
print("=" * 70)

configs = {
    "Tiny": {
        "d_model": 64,
        "num_heads": 4,
        "num_layers": 2,
        "d_ff": 256,
        "params": "~1M"
    },
    "Small (BERT-Small)": {
        "d_model": 512,
        "num_heads": 8,
        "num_layers": 6,
        "d_ff": 2048,
        "params": "~30M"
    },
    "Base (BERT-Base, GPT-2)": {
        "d_model": 768,
        "num_heads": 12,
        "num_layers": 12,
        "d_ff": 3072,
        "params": "~110M"
    },
    "Large (BERT-Large)": {
        "d_model": 1024,
        "num_heads": 16,
        "num_layers": 24,
        "d_ff": 4096,
        "params": "~340M"
    },
    "XL (GPT-3)": {
        "d_model": 12288,
        "num_heads": 96,
        "num_layers": 96,
        "d_ff": 49152,
        "params": "~175B"
    }
}

for name, config in configs.items():
    print(f"\n{name}:")
    for key, value in config.items():
        print(f"  {key}: {value}")

## 📝 Check Your Understanding

1. What are the main components of a Transformer encoder layer?
2. Why do we need residual connections?
3. What's the difference between self-attention and cross-attention?
4. Why does the decoder use masked attention?
5. What are the three main Transformer variants?

## 🎯 Summary

**Transformer architecture**:
- **Encoder**: Process input (bidirectional)
- **Decoder**: Generate output (autoregressive)
- **No recurrence**: Fully parallel processing

**Key components**:
1. **Multi-head attention**: Multiple attention perspectives
2. **Feed-forward**: Position-wise transformation
3. **Residual connections**: Enable deep networks
4. **Layer normalization**: Stable training
5. **Positional encoding**: Inject position information

**Encoder layer**:
```
x → [Self-Attention + Add & Norm] → [Feed-Forward + Add & Norm] → output
```

**Decoder layer**:
```
x → [Masked Self-Attn + Add & Norm] 
  → [Cross-Attn + Add & Norm] 
  → [Feed-Forward + Add & Norm] 
  → output
```

**Three variants**:
1. **Encoder-Decoder**: Translation (T5, BART)
2. **Encoder-Only**: Understanding (BERT)
3. **Decoder-Only**: Generation (GPT)

**Why Transformers won**:
- Parallel processing (faster)
- Long-range dependencies (attention)
- Scalable (billions of parameters)
- Transfer learning (pre-train → fine-tune)

**Next up**: Using pre-trained Transformers with Hugging Face! →